# Benchmark analysis

Load the `N=50` benchmark, compute ranking metrics at 50 and 300, and summarize performance by disease and across diseases. Rankings are restricted to the top 500 candidates for faster exploratory analysis.

In [ ]:
import pickle
import sys
from pathlib import Path
from tqdm import tqdm

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "bioGraph").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

class NumPyCompatibleUnpickler(pickle.Unpickler):
    """Load NumPy 2.x pickles with kernels that still use NumPy 1.x."""

    def find_class(self, module, name):
        if module == "numpy._core" or module.startswith("numpy._core."):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)


results_path = project_root / "outputs" / "results" / "results_N50.pkl"
with results_path.open("rb") as handle:
    benchmark_results = NumPyCompatibleUnpickler(handle).load()

disease_properties_path = project_root / "data" / "processed" / "diseases_prop.pkl"
with disease_properties_path.open("rb") as handle:
    diseases_prop = NumPyCompatibleUnpickler(handle).load()

print(f"Loaded {results_path}")
print(f"Runs: {len(benchmark_results['runs']):,}")
print(f"Loaded properties for {len(diseases_prop):,} diseases from {disease_properties_path}")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from bioGraph import sim
from bioGraph.evaluation.metrics import compute_ranking_metrics

sim.validate_benchmark_results(benchmark_results)

runs = benchmark_results["runs"]
nodelist = np.asarray(benchmark_results["nodelist"])
disease_set = benchmark_results["config"]["disease_set"]
method_set = benchmark_results["config"]["method_set"]
metric_cutoffs = (25, 100)
comparison_cutoffs = (25, 300)
pathway_metric = "recall"  # Use "average_precision" for AP.
pathway_metric_cutoff = 100  # Suggested: Recall@100 or AP@25.
color_threshold = 0.25
computed_cutoffs = tuple(
    sorted(set(metric_cutoffs + comparison_cutoffs + (pathway_metric_cutoff,)))
)
ranking_k = 1000  # Increase later for deeper cutoffs/full-ranking metrics.

if ranking_k < max(computed_cutoffs):
    raise ValueError("ranking_k must be at least the largest metric cutoff.")

print(f"Diseases: {len(disease_set)} | Methods: {len(method_set)}")
print(f"Computed cutoffs: {computed_cutoffs} | Ranking depth: {ranking_k}")

In [ ]:
def top_k_scores_to_ranking(scores, nodelist, seed_genes, k=500):
    """Return only the highest-scoring non-seed genes.

    ``argpartition`` avoids sorting the complete score vector. The selected
    candidates are then ordered by descending score and gene ID, matching
    the deterministic ordering used by ``scores_to_ranking``.
    """
    scores = np.asarray(scores, dtype=float)
    if scores.ndim != 1 or len(scores) != len(nodelist):
        raise ValueError("scores and nodelist must be one-dimensional and equally long.")

    candidate_mask = ~np.isin(nodelist, list(seed_genes))
    candidate_indices = np.flatnonzero(candidate_mask)
    cutoff = min(int(k), len(candidate_indices))
    if cutoff == 0:
        return []

    candidate_scores = scores[candidate_indices]
    if cutoff < len(candidate_indices):
        selected_local = np.argpartition(candidate_scores, -cutoff)[-cutoff:]
        selected = candidate_indices[selected_local]
    else:
        selected = candidate_indices

    order = np.lexsort((nodelist[selected], -scores[selected]))
    selected = selected[order]
    return [
        {"gene_id": gene_id, "score": float(score)}
        for gene_id, score in zip(nodelist[selected], scores[selected])
    ]

In [ ]:
metric_rows = []

for run in tqdm(runs):
    for method_name in method_set:
        ranking = top_k_scores_to_ranking(
            run["scores"][method_name],
            nodelist,
            run["train_genes"],
            k=ranking_k,
        )
        for cutoff in computed_cutoffs:
            metrics = compute_ranking_metrics(
                ranking, run["test_genes"], k=cutoff
            )
            metric_rows.append(
                {
                    "disease": run["disease"],
                    "seed": run["seed"],
                    "method": method_name,
                    "cutoff": cutoff,
                    **metrics,
                }
            )

run_metrics = pd.DataFrame(metric_rows)
display(run_metrics.head())

## Performance by disease across runs

For every disease and method, report the mean and sample standard deviation across its repeated runs.

In [ ]:
metric_columns = [
    "average_precision",
    "recall",
    "f1_score",
    "roc_auc",
    "precision_recall_auc",
]

def summarize_runs(data, group_columns):
    """Compute mean, sample std (ddof=1), and number of raw runs."""
    summary = (
        data.groupby(group_columns)[metric_columns]
        .agg(["mean", "std"])
        .reset_index()
    )
    summary.columns = [
        "_".join(part for part in column if part)
        if isinstance(column, tuple) else column
        for column in summary.columns
    ]
    run_counts = data.groupby(group_columns).size().rename("n_runs").reset_index()
    return summary.merge(run_counts, on=group_columns, validate="one_to_one")


disease_method_performance = summarize_runs(
    run_metrics, ["disease", "method", "cutoff"]
)
metric_labels = {
    "average_precision": "MAP",
    "recall": "MR",
    "f1_score": "MF1",
    "roc_auc": "MROC-AUC",
    "precision_recall_auc": "MPR-AUC",
}


def format_mean_std(mean_value, std_value, significant_digits=2):
    """Format as mean (std), aligning mean precision with the std."""
    mean_value = float(mean_value)
    std_value = float(std_value)
    if not np.isfinite(std_value):
        return f"{mean_value:g} (n/a)"
    if std_value == 0.0:
        return f"{mean_value:.3f} (0)"

    order = int(np.floor(np.log10(abs(std_value))))
    decimals = max(0, significant_digits - 1 - order)
    std_text = f"{std_value:.{decimals}f}"
    if "." in std_text:
        std_text = std_text.rstrip("0").rstrip(".")
    aligned_decimals = len(std_text.split(".")[1]) if "." in std_text else 0
    mean_text = f"{mean_value:.{aligned_decimals}f}"
    return f"{mean_text} ({std_text})"


def format_summary_table(summary, index_columns, cutoff):
    """Create a presentation table with concise metric labels."""
    selected = summary.query("cutoff == @cutoff").set_index(index_columns)
    formatted = pd.DataFrame(index=selected.index)
    for metric, label in metric_labels.items():
        formatted[f"{label}@{cutoff}"] = [
            format_mean_std(mean_value, std_value)
            for mean_value, std_value in zip(
                selected[f"{metric}_mean"], selected[f"{metric}_std"]
            )
        ]
    formatted["N"] = selected["n_runs"].astype(int)
    return formatted

for cutoff in metric_cutoffs:
    print(f"Mean (standard deviation) across runs at K={cutoff}")
    display(
        format_summary_table(
            disease_method_performance, ["disease", "method"], cutoff
        )
    )

## Breast neoplasms: methods versus performance

Rows are prioritization methods; columns are run-averaged performance scores shown as mean (standard deviation).

In [ ]:
breast_disease = "asthma"#"breast neoplasms"
breast_summary = disease_method_performance.query("disease == @breast_disease")

breast_tables = []
for cutoff in metric_cutoffs:
    cutoff_table = format_summary_table(
        breast_summary, ["method"], cutoff
    ).drop(columns="N")
    breast_tables.append(cutoff_table)

breast_performance_matrix = pd.concat(breast_tables, axis=1)
breast_performance_matrix["N"] = (
    breast_summary[breast_summary["cutoff"] == metric_cutoffs[0]]
    .set_index("method")["n_runs"]
    .astype(int)
)
breast_performance_matrix = breast_performance_matrix.loc[
    [method for method in method_set if method in breast_performance_matrix.index]
]

display(breast_performance_matrix)

## Performance across all diseases and runs

For every method, compute the mean and sample standard deviation directly from all raw disease runs. No intermediate disease means are averaged.

In [ ]:
overall_method_performance = summarize_runs(
    run_metrics, ["method", "cutoff"]
)

for cutoff in metric_cutoffs:
    print(f"Mean (standard deviation) across all diseases and runs at K={cutoff}")
    display(
        format_summary_table(overall_method_performance, ["method"], cutoff)
        .loc[
            overall_method_performance.query("cutoff == @cutoff")
            .sort_values("average_precision_mean", ascending=False)["method"]
        ]
    )

## Compact mean ± standard-deviation tables

The same run-level summaries, formatted for easier comparison of average precision, recall, and F1.

In [ ]:
for cutoff in metric_cutoffs:
    print(f"Per-disease summaries at K={cutoff}")
    display(
        format_summary_table(
            disease_method_performance, ["disease", "method"], cutoff
        )
    )
    print(f"All diseases and runs at K={cutoff}")
    display(
        format_summary_table(overall_method_performance, ["method"], cutoff)
    )

## Breast neoplasms: method comparison at K=25 and K=300

Methods are ordered by mean average precision at K=25. Values are mean (standard deviation) across the matching benchmark runs.

In [ ]:
target_disease = "breast neoplasms"
breast_run_metrics = run_metrics.query("disease == @target_disease").copy()
missing_cutoffs = set(comparison_cutoffs) - set(breast_run_metrics["cutoff"].unique())

# This makes the section robust when run_metrics is still cached from an
# earlier notebook execution that did not yet include K=25. Only the
# missing breast-neoplasms metrics are computed, not all diseases again.
if missing_cutoffs:
    missing_rows = []
    for run in runs:
        if run["disease"] != target_disease:
            continue
        for method_name in method_set:
            ranking = top_k_scores_to_ranking(
                run["scores"][method_name], nodelist, run["train_genes"], k=ranking_k
            )
            for cutoff in sorted(missing_cutoffs):
                metrics = compute_ranking_metrics(ranking, run["test_genes"], k=cutoff)
                missing_rows.append({
                    "disease": target_disease,
                    "seed": run["seed"],
                    "method": method_name,
                    "cutoff": cutoff,
                    **metrics,
                })
    breast_run_metrics = pd.concat(
        [breast_run_metrics, pd.DataFrame(missing_rows)], ignore_index=True
    )

breast_summary = summarize_runs(
    breast_run_metrics, ["disease", "method", "cutoff"]
)
method_order = (
    breast_summary.query("cutoff == 25")
    .sort_values("average_precision_mean", ascending=False)["method"]
    .tolist()
)

breast_tables = []
for cutoff in comparison_cutoffs:
    cutoff_table = format_summary_table(breast_summary, ["method"], cutoff)
    breast_tables.append(cutoff_table.drop(columns="N"))

breast_method_performance = pd.concat(breast_tables, axis=1).loc[method_order]
breast_method_performance["N"] = (
    breast_summary.query("cutoff == 25").set_index("method").loc[method_order, "n_runs"]
)
display(breast_method_performance)

## Breast neoplasms: paired tests against RWR

Two-sided paired t-tests compare every method with RWR across matching random seeds. Tests are performed for every metric at K=25 and K=300.

In [ ]:
from scipy.stats import ttest_rel


def significance_label(p_value):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


test_rows = []
for cutoff in comparison_cutoffs:
    cutoff_data = breast_run_metrics.query("cutoff == @cutoff")
    reference = cutoff_data.query("method == 'RWR'").set_index("seed")
    for metric in metric_columns:
        reference_values = reference[metric].sort_index()
        for method_name in method_set:
            if method_name == "RWR":
                continue
            method_values = (
                cutoff_data.query("method == @method_name")
                .set_index("seed")[metric]
                .reindex(reference_values.index)
            )
            if method_values.isna().any():
                raise ValueError(f"Missing paired runs for {method_name}, {metric}, K={cutoff}.")
            differences = method_values.to_numpy() - reference_values.to_numpy()
            if np.allclose(differences, 0.0):
                t_statistic, p_value = 0.0, 1.0
            else:
                t_statistic, p_value = ttest_rel(
                    method_values, reference_values, nan_policy="raise"
                )
            test_rows.append({
                "K": cutoff,
                "Metric": f"{metric_labels[metric]}@{cutoff}",
                "Method": method_name,
                "Method mean (std)": format_mean_std(method_values.mean(), method_values.std(ddof=1)),
                "RWR mean (std)": format_mean_std(reference_values.mean(), reference_values.std(ddof=1)),
                "Mean difference": float(differences.mean()),
                "t statistic": float(t_statistic),
                "p value": float(p_value),
                "Significance": significance_label(float(p_value)),
                "N pairs": len(differences),
            })

paired_tests_vs_rwr = pd.DataFrame(test_rows)
paired_tests_display = paired_tests_vs_rwr.copy()
paired_tests_display["Mean difference"] = paired_tests_display["Mean difference"].map(
    lambda value: f"{value:+.4g}"
)
paired_tests_display["t statistic"] = paired_tests_display["t statistic"].map(
    lambda value: f"{value:+.3f}"
)
paired_tests_display["p value"] = paired_tests_display["p value"].map(
    lambda value: f"{value:.3g}"
)
display(paired_tests_display.set_index(["K", "Metric", "Method"]))
print("Significance: * p<0.05, ** p<0.01, *** p<0.001; ns = not significant.")

## Mean ranking performance versus pathway density

For disease pathway $H_d=(V_d,E_d)$, density is $2|E_d|/(|V_d|(|V_d|-1))$. Each point represents one disease. Configure `pathway_metric` and `pathway_metric_cutoff` above—for example, Recall@100 or AP@25.

In [ ]:
import math

import matplotlib.pyplot as plt
from scipy.stats import linregress

from bioGraph.data.loading import load_ppi_graph


metric_display_names = {"recall": "Recall", "average_precision": "AP"}
if pathway_metric not in metric_display_names:
    raise ValueError("pathway_metric must be 'recall' or 'average_precision'.")
pathway_metric_name = metric_display_names[pathway_metric]

# Ensure the selected metric/cutoff exists even with stale cached results.
# from an older execution of the notebook.
if pathway_metric_cutoff not in set(run_metrics["cutoff"].unique()):
    pathway_metric_rows = []
    for run in tqdm(runs, desc=f"{pathway_metric_name}@{pathway_metric_cutoff}"):
        for method_name in method_set:
            ranking = top_k_scores_to_ranking(
                run["scores"][method_name],
                nodelist,
                run["train_genes"],
                k=ranking_k,
            )
            metrics = compute_ranking_metrics(
                ranking, run["test_genes"], k=pathway_metric_cutoff
            )
            pathway_metric_rows.append({
                "disease": run["disease"],
                "seed": run["seed"],
                "method": method_name,
                "cutoff": pathway_metric_cutoff,
                **metrics,
            })
    run_metrics = pd.concat(
        [run_metrics, pd.DataFrame(pathway_metric_rows)], ignore_index=True
    )

mean_pathway_performance = (
    run_metrics.query("cutoff == @pathway_metric_cutoff")
    .groupby(["disease", "method"], as_index=False)[pathway_metric]
    .mean()
    .rename(columns={pathway_metric: "mean_pathway_metric"})
)

# Every split partitions the same known graph genes, so their union gives V_d.
disease_genes = {disease: set() for disease in disease_set}
for run in runs:
    disease_genes[run["disease"]].update(run["train_genes"])
    disease_genes[run["disease"]].update(run["test_genes"])

ppi_path = project_root / "data" / "raw" / "PPI202207.txt"
Ggen = load_ppi_graph(ppi_path)
density_rows = []
for disease_name, genes in disease_genes.items():
    pathway_nodes = set(genes) & set(Ggen)
    node_count = len(pathway_nodes)
    edge_count = Ggen.subgraph(pathway_nodes).number_of_edges()
    density = (
        2.0 * edge_count / (node_count * (node_count - 1))
        if node_count > 1 else 0.0
    )
    density_rows.append({
        "disease": disease_name,
        "pathway_density": density,
        "pathway_nodes": node_count,
        "pathway_edges": edge_count,
    })

pathway_density = pd.DataFrame(density_rows)
density_pathway_performance = mean_pathway_performance.merge(
    pathway_density, on="disease", validate="many_to_one"
)

n_columns = 5
n_rows = math.ceil(len(method_set) / n_columns)
fig, axes = plt.subplots(
    n_rows, n_columns, figsize=(4.4 * n_columns, 4.0 * n_rows),
    sharex=True, sharey=True, squeeze=False,
)
rwr_better_diseases = (
    density_pathway_performance.query(
        "method == 'RWR' and mean_pathway_metric > @color_threshold"
    )["disease"].tolist()
)
highlight_cmap = plt.get_cmap("hsv", max(len(rwr_better_diseases), 1))
highlight_colors = {
    disease: highlight_cmap(index)
    for index, disease in enumerate(rwr_better_diseases)
}

regression_rows = []
for plot_index, (axis, method_name) in enumerate(zip(axes.flat, method_set)):
    method_data = density_pathway_performance.query("method == @method_name")
    x_values = method_data["pathway_density"].to_numpy(dtype=float)
    y_values = method_data["mean_pathway_metric"].to_numpy(dtype=float)
    breast_mask = method_data["disease"].eq("breast neoplasms").to_numpy()
    for point_index, disease_name in enumerate(method_data["disease"]):
        if disease_name == "breast neoplasms":
            continue
        axis.scatter(
            x_values[point_index], y_values[point_index],
            color=highlight_colors.get(disease_name, "tab:blue"),
            s=32, alpha=0.75, edgecolor="white", linewidth=0.4,
        )
    axis.scatter(
        x_values[breast_mask], y_values[breast_mask],
        marker="*", color="tab:orange", zorder=5,
    )

    fit = linregress(x_values, y_values)
    x_fit = np.linspace(x_values.min(), x_values.max(), 200)
    axis.plot(
        x_fit, fit.intercept + fit.slope * x_fit,
        color="tab:red", linewidth=1.6,
    )
    fit_text = (
        f"m = {format_mean_std(fit.slope, fit.stderr)}\n"
        f"b = {format_mean_std(fit.intercept, fit.intercept_stderr)}\n"
        f"R² = {fit.rvalue ** 2:.2g}"
    )
    axis.text(
        0.97, 0.97, fit_text, transform=axis.transAxes,
        ha="right", va="top", fontsize=8.5,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8, "edgecolor": "0.8"},
    )
    regression_rows.append({
        "method": method_name,
        "slope": fit.slope,
        "slope_stderr": fit.stderr,
        "intercept": fit.intercept,
        "intercept_stderr": fit.intercept_stderr,
        "r_squared": fit.rvalue ** 2,
        "p_value": fit.pvalue,
    })

    axis.set_title(method_name)
    axis.grid(alpha=0.25)
    axis.tick_params(axis="both", labelbottom=True, labelleft=True)
    row_index, column_index = divmod(plot_index, n_columns)
    if row_index == n_rows - 1:
        axis.set_xlabel("Pathway density")
    if column_index == 0:
        axis.set_ylabel(f"Mean {pathway_metric_name}@{pathway_metric_cutoff}")

for axis in axes.flat[len(method_set):]:
    axis.set_visible(False)

pathway_density_regressions = pd.DataFrame(regression_rows)

fig.suptitle(
    f"Mean {pathway_metric_name}@{pathway_metric_cutoff} vs pathway density",
    fontsize=15, y=1.01,
)
fig.tight_layout()
plt.show()

display(density_pathway_performance.head())

## AP@25 and Recall@100 versus disease pathway properties

The table reports both fixed metrics. Eight figures (two metrics × four properties) are displayed and exported to one multipage PDF.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import linregress


fixed_metrics = [
    {"metric": "average_precision", "cutoff": 25, "label": "AP@25"},
    {"metric": "recall", "cutoff": 100, "label": "Recall@100"},
]
property_specs = [
    ("pathway_density", "Pathway density"),
    ("largest_component_fraction", "Largest connected component fraction"),
    ("mean_component_distance", "Mean component distance"),
    ("conductance", "Conductance"),
]

required_cutoffs = {spec["cutoff"] for spec in fixed_metrics}
missing_cutoffs = required_cutoffs - set(run_metrics["cutoff"].unique())
if missing_cutoffs:
    raise RuntimeError(
        f"run_metrics is missing K={sorted(missing_cutoffs)}. "
        "Rerun the metric-computation cell above once."
    )

# One method table containing exactly AP@25 and Recall@100.
table_parts = []
for spec in fixed_metrics:
    cutoff = spec["cutoff"]
    values = run_metrics.query("cutoff == @cutoff")
    summary = values.groupby("method")[spec["metric"]].agg(["mean", "std"])
    table_parts.append(pd.Series(
        [format_mean_std(mean, std) for mean, std in zip(summary["mean"], summary["std"])],
        index=summary.index, name=spec["label"],
    ))
performance_ap25_recall100 = pd.concat(table_parts, axis=1)
ap25_order = (
    run_metrics.query("cutoff == 25").groupby("method")["average_precision"]
    .mean().sort_values(ascending=False).index
)
performance_ap25_recall100 = performance_ap25_recall100.loc[ap25_order]
display(performance_ap25_recall100)

property_rows = []
for disease_name, record in diseases_prop.items():
    property_rows.append({
        "disease": disease_name,
        **{property_name: record.get(property_name, np.nan) for property_name, _ in property_specs},
    })
disease_property_table = pd.DataFrame(property_rows)

reports_dir = project_root / "outputs" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
property_pdf_path = reports_dir / "AP25_Recall100_vs_disease_properties.pdf"
property_regression_rows = []

with PdfPages(property_pdf_path) as pdf:
    for metric_spec in fixed_metrics:
        metric_name = metric_spec["metric"]
        metric_cutoff = metric_spec["cutoff"]
        metric_label = metric_spec["label"]
        disease_method_means = (
            run_metrics.query("cutoff == @metric_cutoff")
            .groupby(["disease", "method"], as_index=False)[metric_name].mean()
            .rename(columns={metric_name: "mean_performance"})
            .merge(disease_property_table, on="disease", validate="many_to_one")
        )
        highlighted_diseases = (
            disease_method_means.query(
                "method == 'RWR' and mean_performance > @color_threshold"
            )["disease"].tolist()
        )
        color_map = plt.get_cmap("hsv", max(len(highlighted_diseases), 1))
        disease_colors = {
            disease: color_map(index)
            for index, disease in enumerate(highlighted_diseases)
        }

        for property_name, property_label in property_specs:
            fig, axes = plt.subplots(
                2, 5, figsize=(22, 8), sharex=True, sharey=True, squeeze=False
            )
            for plot_index, (axis, method_name) in enumerate(zip(axes.flat, method_set)):
                method_data = disease_method_means.query("method == @method_name").copy()
                method_data = method_data.loc[
                    np.isfinite(method_data[property_name])
                    & np.isfinite(method_data["mean_performance"])
                ]
                for _, point in method_data.iterrows():
                    if point["disease"] == "breast neoplasms":
                        marker, color, size, zorder = "*", "tab:orange", 36, 5
                    else:
                        marker, size, zorder = "o", 32, 2
                        color = disease_colors.get(point["disease"], "tab:blue")
                    axis.scatter(
                        point[property_name], point["mean_performance"],
                        marker=marker, color=color, s=size, alpha=0.8,
                        edgecolor="white", linewidth=0.4, zorder=zorder,
                    )

                x_values = method_data[property_name].to_numpy(dtype=float)
                y_values = method_data["mean_performance"].to_numpy(dtype=float)
                if len(x_values) >= 3 and np.ptp(x_values) > 0:
                    fit = linregress(x_values, y_values)
                    x_fit = np.linspace(x_values.min(), x_values.max(), 200)
                    axis.plot(x_fit, fit.intercept + fit.slope * x_fit, color="tab:red", linewidth=1.6)
                    axis.text(
                        0.97, 0.97,
                        f"m = {format_mean_std(fit.slope, fit.stderr)}\n"
                        f"b = {format_mean_std(fit.intercept, fit.intercept_stderr)}\n"
                        f"R² = {fit.rvalue ** 2:.2g}",
                        transform=axis.transAxes, ha="right", va="top", fontsize=8.5,
                        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white",
                              "alpha": 0.8, "edgecolor": "0.8"},
                    )
                    property_regression_rows.append({
                        "metric": metric_label, "property": property_name,
                        "method": method_name, "n_diseases": len(x_values),
                        "slope": fit.slope, "slope_stderr": fit.stderr,
                        "intercept": fit.intercept,
                        "intercept_stderr": fit.intercept_stderr,
                        "r_squared": fit.rvalue ** 2, "p_value": fit.pvalue,
                    })

                row_index, column_index = divmod(plot_index, 5)
                axis.set_title(method_name)
                axis.grid(alpha=0.25)
                axis.tick_params(axis="both", labelbottom=True, labelleft=True)
                if row_index == 1:
                    axis.set_xlabel(property_label)
                if column_index == 0:
                    axis.set_ylabel(f"Mean {metric_label}")

            fig.suptitle(f"{metric_label} vs {property_label}", fontsize=15, y=1.01)
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.show()

property_regressions = pd.DataFrame(property_regression_rows)
print(f"Saved 8 figures to {property_pdf_path}")

,AP@25,Recall@100
method,,
QA*,0.028 (0.065),0.16 (0.19)
DIAMOND,0.02 (0.07),0.13 (0.17)
RWR,0.020 (0.058),0.14 (0.19)
QA1,0.018 (0.047),0.15 (0.19)
aNBR,0.017 (0.053),0.12 (0.17)
QA0,0.014 (0.041),0.13 (0.18)
DK*,0.013 (0.034),0.11 (0.15)
DK,0.012 (0.033),0.11 (0.15)
GCN,0.011 (0.044),0.06 (0.12)


NameError: name 'diseases_prop' is not defined